# release16_nonconvex_checkpoint_training_contract_v1

Purpose: convert the current nonconvex checkpoint-objective readiness evidence plus the failed release-gate BC official replay into a concrete, non-promoting training contract.

Governance:
- **Baselines**: canonical `soxlbacktest_qlib_walkforward_baseline_v1`; direct comparison `release16_release_gate_bc_official_replay_v1`; source readiness card `release16_nonconvex_checkpoint_objective_readiness_v1`.
- **Role**: `diagnose` / training handoff. This card does not train, retain, or promote an actor.
- **Target window**: `wf_2020`, `wf_2022`, `wf_2023_2024`. Accepted trade-off is none for retention: each fold still requires relative return >= 30pp and MaxDD < 20%.
- **Changed layer**: `model_selection`, `checkpoint_objective`, `training_contract`, `slice_guard`.
- **External reference**: official qlib daily-path replay and SOXLBacktest qlib walk-forward baseline.
- **Hypothesis**: the release64 BC slice repair is not sufficient because official path replay failed; the next useful run must optimize checkpoint/reward selection against fold-weighted official replay gaps and sparse-slice guards before PPO continuation.
- **Acceptance criteria**: release64 is excluded from retention, objective terms and fold blockers are machine-readable, and frontier advances from the previous readiness card to this explicit training contract without promotion.
- **Failure mode**: if a later training run fixes supervised slices but not official daily-path return/drawdown, it must be rejected before actor retention.
- **Comparison-table row key**: `release16_nonconvex_checkpoint_training_contract_v1`.


In [1]:
from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "SPEC.md").exists():
        ROOT = candidate
        break

CARD = ROOT / "runtime/model_selection/card/release16_nonconvex_checkpoint_training_contract_v1"
CARD.mkdir(parents=True, exist_ok=True)

SOURCE_READY = ROOT / "runtime/model_selection/card/release16_nonconvex_checkpoint_objective_readiness_v1"
SOURCE_BC_SCREEN = ROOT / "runtime/model_selection/card/release16_release_gate_bc_weight_screen_v1"
SOURCE_OFFICIAL = ROOT / "runtime/model_selection/card/release16_release_gate_bc_official_replay_v1"
RETENTION_GATE = ROOT / "runtime/model_selection/card/model_retention_walkforward_gate_v1/summary.json"

WINDOWS = ["wf_2020", "wf_2022", "wf_2023_2024"]
EXPERIMENT = "release16_nonconvex_checkpoint_training_contract_v1"

def read_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))

def write_json(path: Path, payload: dict) -> None:
    path.write_text(json.dumps(payload, indent=2, sort_keys=True) + chr(10), encoding="utf-8")

ready_summary = read_json(SOURCE_READY / "summary.json")
official_summary = read_json(SOURCE_OFFICIAL / "summary.json")
retention_summary = read_json(RETENTION_GATE)
selected_bc_metrics = pd.read_csv(SOURCE_OFFICIAL / "bc_official_replay_metrics.csv")
selected_bc_metrics = selected_bc_metrics[selected_bc_metrics["candidate"].eq(official_summary["selected_config"])].copy()
bc_slice_metrics = pd.read_csv(SOURCE_BC_SCREEN / "config_slice_metrics.csv")
bc_fold_metrics = pd.read_csv(SOURCE_BC_SCREEN / "selected_config_fold_metrics.csv")
ready_slice = pd.read_csv(SOURCE_READY / "state_head_slice_precondition.csv")
ready_teacher = pd.read_csv(SOURCE_READY / "selected_fold_teacher_metrics.csv")
ready_objective = pd.read_csv(SOURCE_READY / "checkpoint_objective_scores.csv")
print("source_ready", ready_summary["decision"])
print("source_official", official_summary["decision"])
print("selected_bc_config", official_summary["selected_config"])


source_ready release16_nonconvex_checkpoint_objective_readiness_ready_no_actor_retention
source_official release_gate_weighted_bc_official_replay_failed_no_promotion
selected_bc_config release64_no_objective


In [2]:
selected_config = official_summary["selected_config"]
release_slice = bc_slice_metrics[
    bc_slice_metrics["config_id"].eq(selected_config)
    & bc_slice_metrics["slice"].eq("release_gate_state")
][["fold", "samples", "accuracy", "loss", "predicted_unique_actions"]].rename(
    columns={
        "samples": "release_slice_samples",
        "accuracy": "release_slice_accuracy",
        "loss": "release_slice_loss",
        "predicted_unique_actions": "release_slice_predicted_unique_actions",
    }
)
fold_gate = bc_fold_metrics[["fold", "required_gate", "failed_required_slices", "all_accuracy", "release_accuracy"]].rename(
    columns={
        "required_gate": "release64_required_slice_gate",
        "failed_required_slices": "release64_failed_required_slices",
        "all_accuracy": "release64_all_accuracy",
        "release_accuracy": "release64_release_accuracy",
    }
)
official_sel = selected_bc_metrics[[
    "fold", "return_pct", "relative_return_pp", "max_dd_pct", "avg_pos", "wf_gate_20dd", "strict_gate_15dd"
]].rename(columns={
    "return_pct": "release64_official_return_pct",
    "relative_return_pp": "release64_official_relative_return_pp",
    "max_dd_pct": "release64_official_max_dd_pct",
    "avg_pos": "release64_official_avg_pos",
    "wf_gate_20dd": "release64_official_wf_gate_20dd",
    "strict_gate_15dd": "release64_official_strict_gate_15dd",
})
teacher = ready_teacher[["window", "return_pct", "relative_return_pct", "max_dd_pct", "avg_pos", "candidate", "family", "candidate_gate"]].rename(
    columns={
        "window": "fold",
        "return_pct": "teacher_return_pct",
        "relative_return_pct": "teacher_relative_return_pp",
        "max_dd_pct": "teacher_max_dd_pct",
        "avg_pos": "teacher_avg_pos",
        "candidate": "teacher_candidate",
        "family": "teacher_family",
        "candidate_gate": "teacher_candidate_gate",
    }
)
slice_pre = ready_slice.rename(columns={"slice_precondition_gate": "state_head_slice_precondition_gate"})
reconciliation = (
    pd.DataFrame({"fold": WINDOWS})
    .merge(fold_gate, on="fold", how="left")
    .merge(release_slice, on="fold", how="left")
    .merge(official_sel, on="fold", how="left")
    .merge(slice_pre, on="fold", how="left")
    .merge(teacher, on="fold", how="left")
)
reconciliation["release64_supervised_slice_pass_but_official_fail"] = (
    reconciliation["release64_required_slice_gate"].fillna(False).astype(bool)
    & ~reconciliation["release64_official_wf_gate_20dd"].fillna(False).astype(bool)
)
reconciliation["required_next_guard"] = np.where(
    reconciliation["release64_supervised_slice_pass_but_official_fail"],
    "official_path_gate_and_drawdown_drift_stop",
    "slice_fit_repair",
)
reconciliation.to_csv(CARD / "slice_guard_official_replay_reconciliation.csv", index=False)
reconciliation


,fold,release64_required_slice_gate,release64_failed_required_slices,release64_all_accuracy,release64_release_accuracy,release_slice_samples,release_slice_accuracy,release_slice_loss,release_slice_predicted_unique_actions,release64_official_return_pct,...,state_head_slice_precondition_gate,teacher_return_pct,teacher_relative_return_pp,teacher_max_dd_pct,teacher_avg_pos,teacher_candidate,teacher_family,teacher_candidate_gate,release64_supervised_slice_pass_but_official_fail,required_next_guard
0,wf_2020,True,NaN,0.652493,0.906250,32,0.906250,0.999740,4,80.375210,...,False,79.426700,21.333130,16.652151,0.307464,dd_fallback_0.075_10d,drawdown_fallback,False,True,official_path_gate_and_drawdown_drift_stop
1,wf_2022,True,NaN,0.754941,1.000000,8,1.000000,0.904931,4,-23.837432,...,False,-14.345198,72.242874,15.341329,0.086368,static_blend_state_0.80,static_blend,True,True,official_path_gate_and_drawdown_drift_stop
2,wf_2023_2024,True,NaN,0.707672,0.818182,11,0.818182,1.088289,2,13.945428,...,False,22.429328,22.785963,5.546722,0.097579,position_cap_0.20,position_cap_fallback,False,True,official_path_gate_and_drawdown_drift_stop


In [3]:
official_failure = selected_bc_metrics.copy()
official_failure["target_relative_return_pp"] = 30.0
official_failure["target_max_dd_pct"] = float(retention_summary.get("max_walkforward_drawdown_pct", 20.0))
official_failure["relative_gap_pp"] = (official_failure["target_relative_return_pp"] - official_failure["relative_return_pp"]).clip(lower=0.0)
official_failure["drawdown_gap_pp"] = (official_failure["max_dd_pct"] - official_failure["target_max_dd_pct"]).clip(lower=0.0)
official_failure["failed_gate_reason"] = official_failure.apply(
    lambda r: ",".join(
        reason for reason, failed in [
            ("relative_return", r["relative_gap_pp"] > 0),
            ("drawdown", r["drawdown_gap_pp"] > 0),
        ] if failed
    ) or "",
    axis=1,
)
official_failure.to_csv(CARD / "release64_official_replay_failure_by_fold.csv", index=False)
official_failure[["fold", "return_pct", "relative_return_pp", "max_dd_pct", "avg_pos", "relative_gap_pp", "drawdown_gap_pp", "failed_gate_reason"]]


,fold,return_pct,relative_return_pp,max_dd_pct,avg_pos,relative_gap_pp,drawdown_gap_pp,failed_gate_reason
1,wf_2020,80.375210,22.281639,21.106156,0.425202,7.718361,1.106156,"relative_return,drawdown"
3,wf_2022,-23.837432,62.750639,26.860565,0.144199,0.000000,6.860565,drawdown
5,wf_2023_2024,13.945428,14.302062,13.959434,0.197766,15.697938,0.000000,relative_return


In [4]:
objective_terms = pd.DataFrame([
    {
        "term": "fold_weighted_relative_return_gap",
        "applies_to": ",".join(WINDOWS),
        "formula": "sum(window_weight * max(0, 30 - official_relative_return_pp))",
        "reason": "wf_2020 and wf_2023_2024 still miss the required relative return even after slice repair",
        "hard_gate": True,
    },
    {
        "term": "drawdown_drift_stop_loss",
        "applies_to": ",".join(WINDOWS),
        "formula": "sum(window_weight * max(0, official_max_dd_pct - 20))",
        "reason": "release64 official replay breaches MaxDD on wf_2020 and wf_2022",
        "hard_gate": True,
    },
    {
        "term": "state_head_slice_fit_guard",
        "applies_to": ",".join(WINDOWS),
        "formula": "block PPO continuation until required sparse slices pass fold-local fit gates",
        "reason": "readiness card still has state_head_slice_precondition_gate=false",
        "hard_gate": True,
    },
    {
        "term": "release64_branch_exclusion",
        "applies_to": selected_config,
        "formula": "exclude selected release-gate BC weighting branch from actor retention",
        "reason": "supervised slice repair did not transfer to official daily path gates",
        "hard_gate": True,
    },
    {
        "term": "official_daily_path_metric_basis",
        "applies_to": ",".join(WINDOWS),
        "formula": "score candidates from official_daily_paths, not raw backtest-only metrics",
        "reason": "previous screens recorded metric-basis mismatch risk",
        "hard_gate": True,
    },
])
objective_terms.to_csv(CARD / "training_objective_terms.csv", index=False)
objective_terms


,term,applies_to,formula,reason,hard_gate
0,fold_weighted_relative_return_gap,"wf_2020,wf_2022,wf_2023_2024","sum(window_weight * max(0, 30 - official_relat...",wf_2020 and wf_2023_2024 still miss the requir...,True
1,drawdown_drift_stop_loss,"wf_2020,wf_2022,wf_2023_2024","sum(window_weight * max(0, official_max_dd_pct...",release64 official replay breaches MaxDD on wf...,True
2,state_head_slice_fit_guard,"wf_2020,wf_2022,wf_2023_2024",block PPO continuation until required sparse s...,readiness card still has state_head_slice_prec...,True
3,release64_branch_exclusion,release64_no_objective,exclude selected release-gate BC weighting bra...,supervised slice repair did not transfer to of...,True
4,official_daily_path_metric_basis,"wf_2020,wf_2022,wf_2023_2024","score candidates from official_daily_paths, no...",previous screens recorded metric-basis mismatc...,True


In [5]:
best_existing = dict(ready_summary.get("best_existing_single_candidate", {}))
metrics = []
for row in ready_summary.get("metrics", []):
    metrics.append(dict(row))

selected_official_fail_folds = official_failure.loc[~official_failure["wf_gate_20dd"].astype(bool), "fold"].tolist()
state_slice_failed_folds = ready_slice.loc[~ready_slice["slice_precondition_gate"].astype(bool), "fold"].tolist()
release64_supervised_all_pass = bool(bc_fold_metrics["required_gate"].all())
release64_official_all_pass = bool(selected_bc_metrics["wf_gate_20dd"].all())
contract_ready = bool(
    ready_summary.get("target_contract_gate")
    and ready_summary.get("bc_readiness_gate")
    and ready_summary.get("bc_screen_allowed_next")
    and release64_supervised_all_pass
    and not release64_official_all_pass
)
next_mechanism = "train_nonconvex_checkpoint_objective_with_release_slice_guard_and_drawdown_drift_stop_then_official_three_fold_replay"
summary = {
    "experiment": EXPERIMENT,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "decision": "release16_nonconvex_checkpoint_training_contract_ready_no_actor_retention" if contract_ready else "release16_nonconvex_checkpoint_training_contract_incomplete_no_promotion",
    "role": "diagnose",
    "canonical_baseline": "soxlbacktest_qlib_walkforward_baseline_v1",
    "reference_baseline": "soxlbacktest_qlib_walkforward_baseline_v1",
    "direct_comparison_baseline": "release16_release_gate_bc_official_replay_v1",
    "source_card": "release16_nonconvex_checkpoint_objective_readiness_v1",
    "source_official_replay_experiment": "release16_release_gate_bc_official_replay_v1",
    "source_bc_weight_screen_experiment": "release16_release_gate_bc_weight_screen_v1",
    "changed_layer": ["model_selection", "checkpoint_objective", "training_contract", "slice_guard"],
    "target_window": WINDOWS,
    "target_relative_return_pp": 30.0,
    "target_relative_return_by_window_pp": {window: 30.0 for window in WINDOWS},
    "target_max_dd_pct": float(retention_summary.get("max_walkforward_drawdown_pct", 20.0)),
    "drawdown_gate_operator": "<",
    "metric_basis": "training_contract_from_official_daily_path_failure_and_nonconvex_readiness_no_actor",
    "target_contract_gate": bool(ready_summary.get("target_contract_gate")),
    "bc_readiness_gate": bool(ready_summary.get("bc_readiness_gate")),
    "bc_screen_allowed_next": bool(ready_summary.get("bc_screen_allowed_next")),
    "release64_supervised_slice_gate": release64_supervised_all_pass,
    "release64_official_replay_gate": release64_official_all_pass,
    "release64_excluded_from_retention": True,
    "state_head_slice_precondition_gate": bool(ready_summary.get("state_head_slice_precondition_gate")),
    "state_head_slice_failed_folds": state_slice_failed_folds,
    "official_replay_failed_folds": selected_official_fail_folds,
    "training_contract_ready": contract_ready,
    "training_allowed_from_this_card": contract_ready,
    "next_training_allowed_after_objective_screen": contract_ready,
    "official_replay_required_next": True,
    "ppo_allowed_next": False,
    "promotion_allowed": False,
    "actor_retention_allowed": False,
    "primary_blocker": "training_contract_ready" if contract_ready else "bc_candidate_or_slice_gate_failed",
    "next_mechanism": next_mechanism,
    "best_existing_single_candidate": best_existing,
    "metrics": metrics,
    "min_wf_relative_return_pp": ready_summary.get("min_wf_relative_return_pp"),
    "worst_wf_max_drawdown_pct": ready_summary.get("worst_wf_max_drawdown_pct"),
    "max_relative_gap_to_target_pp": ready_summary.get("max_relative_gap_to_target_pp"),
    "max_drawdown_gap_to_target_pp": ready_summary.get("max_drawdown_gap_to_target_pp"),
    "selected_bc_config": selected_config,
    "selected_bc_official_min_relative_return_pp": float(selected_bc_metrics["relative_return_pp"].min()),
    "selected_bc_official_worst_max_dd_pct": float(selected_bc_metrics["max_dd_pct"].max()),
    "selected_bc_official_gate_count_20dd": int(selected_bc_metrics["wf_gate_20dd"].sum()),
    "selected_bc_official_fold_count": int(len(selected_bc_metrics)),
    "objective_terms": objective_terms.to_dict(orient="records"),
    "output_paths": {
        "summary": str((CARD / "summary.json").relative_to(ROOT)),
        "selection": str((CARD / "selection.json").relative_to(ROOT)),
        "training_contract": str((CARD / "training_contract.json").relative_to(ROOT)),
        "training_objective_terms": str((CARD / "training_objective_terms.csv").relative_to(ROOT)),
        "slice_guard_official_replay_reconciliation": str((CARD / "slice_guard_official_replay_reconciliation.csv").relative_to(ROOT)),
        "release64_official_replay_failure_by_fold": str((CARD / "release64_official_replay_failure_by_fold.csv").relative_to(ROOT)),
    },
    "summary_points": [
        "release64_no_objective passes supervised required slice gates but fails official daily-path gates on all three folds, so it is excluded from retention.",
        "The current readiness card still needs a nonconvex checkpoint/reward objective because the best retained-like teacher mix misses the 30pp relative-return floor on wf_2020 and wf_2023_2024.",
        "Next training must use official daily-path metrics, drawdown drift stop loss, and sparse slice guards before any PPO continuation or actor retention.",
    ],
}
selection = {
    "experiment": EXPERIMENT,
    "decision": summary["decision"],
    "source_card": summary["source_card"],
    "direct_comparison_baseline": summary["direct_comparison_baseline"],
    "selected_bc_config": selected_config,
    "training_contract_ready": contract_ready,
    "next_mechanism": next_mechanism,
    "target_contract_gate": summary["target_contract_gate"],
    "bc_readiness_gate": summary["bc_readiness_gate"],
    "bc_screen_allowed_next": summary["bc_screen_allowed_next"],
    "promotion_allowed": False,
    "actor_retention_allowed": False,
}
training_contract = {
    "experiment": EXPERIMENT,
    "source_card": summary["source_card"],
    "direct_comparison_baseline": summary["direct_comparison_baseline"],
    "allowed_next_action": next_mechanism,
    "must_not_retain_actor_until": [
        "all official wf windows have relative_return_pp >= 30",
        "all official wf windows have MaxDD < 20",
        "required sparse slice fit gates pass",
        "dashboard replay evidence exists for any retained actor",
    ],
    "excluded_branches": [
        {
            "branch": selected_config,
            "reason": "official_daily_path gates failed after supervised slice repair",
            "failed_folds": selected_official_fail_folds,
        }
    ],
    "objective_terms": summary["objective_terms"],
}
write_json(CARD / "summary.json", summary)
write_json(CARD / "selection.json", selection)
write_json(CARD / "training_contract.json", training_contract)
summary


{'experiment': 'release16_nonconvex_checkpoint_training_contract_v1',
 'created_at': '2026-06-15T00:28:30.534743+00:00',
 'decision': 'release16_nonconvex_checkpoint_training_contract_ready_no_actor_retention',
 'role': 'diagnose',
 'canonical_baseline': 'soxlbacktest_qlib_walkforward_baseline_v1',
 'reference_baseline': 'soxlbacktest_qlib_walkforward_baseline_v1',
 'direct_comparison_baseline': 'release16_release_gate_bc_official_replay_v1',
 'source_card': 'release16_nonconvex_checkpoint_objective_readiness_v1',
 'source_official_replay_experiment': 'release16_release_gate_bc_official_replay_v1',
 'source_bc_weight_screen_experiment': 'release16_release_gate_bc_weight_screen_v1',
 'changed_layer': ['model_selection',
  'checkpoint_objective',
  'training_contract',
  'slice_guard'],
 'target_window': ['wf_2020', 'wf_2022', 'wf_2023_2024'],
 'target_relative_return_pp': 30.0,
 'target_relative_return_by_window_pp': {'wf_2020': 30.0,
  'wf_2022': 30.0,
  'wf_2023_2024': 30.0},
 'target

## Conclusion

This card is the executable conclusion for the training-contract handoff. It intentionally does **not** retain an actor. The release64 BC weighting branch is rejected for retention because official qlib daily-path replay failed after supervised slice repair. The next valid model loop is a nonconvex checkpoint/reward objective with official-path return/drawdown terms and sparse-slice guards, followed by the required three-fold official replay before any PPO continuation or dashboard retention.
